# Yet New — re-export the layer labeler to GGUF (Kaggle, GPU T4)

The training run (`train_yetnew_layers.ipynb`, 2026-10-24) learned well — **70.9 % on the 774 held-out layers on the GPU** — but its GGUF answered with rambling text in Ollama: the merge/convert step through Unsloth went wrong (its log warned about a modified llama.cpp converter). This notebook does only the export, a plainer way, **without retraining**:

1. the original **Qwen/Qwen2.5-3B-Instruct** in 16-bit + the trained LoRA adapter → merged (PEFT `merge_and_unload`);
2. **checked**: the merged model must answer labels on held-out layers (≥ 55 % on 40, every answer a label) — otherwise it stops, nothing is exported;
3. converted with llama.cpp's own `convert_hf_to_gguf.py`, then quantized to **Q4_K_M** with `llama-quantize`.

**Inputs** (+ Add Input → Upload, **Private**): `yetnew-layers-lora.zip` (the adapter from the training run) and `yetnew-layers-dataset-2026-10-24.zip` (for the check). Settings: **GPU T4 x2**, Internet **on**. Then **Save Version → Save & Run All**. Output: `yetnew-layers.Q4_K_M.gguf`. Delete both datasets afterwards.

Not verified before running (no GPU where this was written): llama.cpp is cloned at its latest version; if its build or converter changed, the error says where.

In [ ]:
# Kaggle ships torchao 0.10, which PEFT 0.21 refuses to load alongside; this export doesn't use it.
!pip uninstall -y -q torchao
!pip install -q "transformers==5.5.0" "peft==0.21.0" "accelerate==1.15.0" "gguf==0.19.0" sentencepiece
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| GPU", torch.cuda.is_available())

In [ ]:
import pathlib, zipfile, json
inp = pathlib.Path("/kaggle/input")
zips = sorted(inp.rglob("*.zip"))
for z in zips:
    zipfile.ZipFile(z).extractall(pathlib.Path("/tmp") / z.stem)
def find(name):
    hits = sorted(p for p in list(inp.rglob(name)) + list(pathlib.Path("/tmp").rglob(name)))
    assert hits, f"{name} not found: add both zips as inputs"
    return hits[0]
ADAPTER = find("adapter_config.json").parent
VAL = find("val.jsonl")
print("adapter:", ADAPTER, "| val:", VAL)
val_rows = [json.loads(l) for l in VAL.read_text(encoding="utf-8").splitlines() if l.strip()]
LABELS = sorted({r["label"] for r in val_rows} | {"wall","door","window","stairs","elevator","restroom","room","walkable","vegetation","furniture","text","dimension","ignore","unknown"})
print(len(val_rows), "held-out layers")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-3B-Instruct"   # the full-precision model the 4-bit training base was made from
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="cuda" if torch.cuda.is_available() else None)
model = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
model.eval()
print("merged")

In [ ]:
# The check that the training notebook's export lacked: the MERGED model must answer labels.
import random
random.seed(0)
sample = random.sample([r for r in val_rows if not r.get("augmented")], 40)
right, odd = 0, []
for r in sample:
    prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=6, do_sample=False)
    got = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    right += got == r["label"]
    if got not in LABELS: odd.append(got[:40])
print(f"merged model: {right}/40 right; answers that aren't a label: {odd[:5]}")
assert not odd, "The merged model doesn't answer with a label — stop: the merge is wrong, nothing is exported."
assert right >= 22, "Below 55 % on 40 held-out layers (the GPU run had 70.9 %) — stop, something is off."
model.save_pretrained("/tmp/merged", safe_serialization=True)
tokenizer.save_pretrained("/tmp/merged")
print("saved /tmp/merged")

In [ ]:
%%bash
set -e
cd /tmp
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp
echo "llama.cpp commit: $(git rev-parse --short HEAD)"
cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
cmake --build build --target llama-quantize -j 4 > /dev/null
ls build/bin/llama-quantize

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH="/tmp/llama.cpp/gguf-py")
subprocess.run([sys.executable, "/tmp/llama.cpp/convert_hf_to_gguf.py", "/tmp/merged", "--outtype", "f16", "--outfile", "/tmp/f16.gguf"], check=True, env=env)
subprocess.run(["/tmp/llama.cpp/build/bin/llama-quantize", "/tmp/f16.gguf", "yetnew-layers.Q4_K_M.gguf", "Q4_K_M"], check=True)
print("written", os.path.getsize("yetnew-layers.Q4_K_M.gguf") / 1e9, "GB")

In [ ]:
# Ollama renders chats with the template embedded in the GGUF: it must render the training prompt.
from gguf import GGUFReader
from jinja2.sandbox import ImmutableSandboxedEnvironment
embedded = GGUFReader("yetnew-layers.Q4_K_M.gguf").fields["tokenizer.chat_template"].contents()
render = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True).from_string(embedded).render
def expected(messages):
    return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages) + "<|im_start|>assistant\n"
for r in val_rows[:5]:
    assert render(messages=r["messages"][:2], add_generation_prompt=True, tools=None) == expected(r["messages"][:2])
print("Embedded chat template renders the training prompt ✓")
print("Done. Download yetnew-layers.Q4_K_M.gguf from this version's Output, then delete both input datasets.")